In [ ]:
#2.11
!pip install scikit-fuzzy -q
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

class GrabFuzzySystem:
    def __init__(self):
        self.setup_system()

    def setup_system(self):
        # Biến đầu vào
        self.khach_hang = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'khach_hang')
        self.dung_gio = ctrl.Antecedent(np.arange(-15, 16, 1), 'dung_gio')
        self.khoang_cach = ctrl.Antecedent(np.arange(0, 31, 1), 'khoang_cach')
        self.giao_thong = ctrl.Antecedent(np.arange(0, 11, 1), 'giao_thong')
        self.thoi_tiet = ctrl.Antecedent(np.arange(0, 11, 1), 'thoi_tiet')

        self.diem_thuong = ctrl.Consequent(np.arange(0, 101, 1), 'diem_thuong')

        self.khach_hang['kem'] = fuzz.trimf(self.khach_hang.universe, [0, 0, 2.5])
        self.khach_hang['trung_binh'] = fuzz.trimf(self.khach_hang.universe, [1.5, 3, 4.5])
        self.khach_hang['tot'] = fuzz.trimf(self.khach_hang.universe, [3.5, 5, 5])

        self.dung_gio['som'] = fuzz.trapmf(self.dung_gio.universe, [-15, -15, -5, 0])
        self.dung_gio['dung_gio'] = fuzz.trimf(self.dung_gio.universe, [-3, 0, 3])
        self.dung_gio['tre'] = fuzz.trapmf(self.dung_gio.universe, [0, 5, 15, 15])

        self.khoang_cach['ngan'] = fuzz.trimf(self.khoang_cach.universe, [0, 0, 8])
        self.khoang_cach['trung_binh'] = fuzz.trimf(self.khoang_cach.universe, [5, 10, 15])
        self.khoang_cach['dai'] = fuzz.trimf(self.khoang_cach.universe, [12, 18, 25])
        self.khoang_cach['rat_xa'] = fuzz.trapmf(self.khoang_cach.universe, [20, 25, 30, 30])

        self.giao_thong['trung_binh'] = fuzz.trimf(self.giao_thong.universe, [0, 3, 6])
        self.giao_thong['cao'] = fuzz.trapmf(self.giao_thong.universe, [5, 8, 10, 10])

        self.thoi_tiet['trung_binh'] = fuzz.trimf(self.thoi_tiet.universe, [0, 3, 6])
        self.thoi_tiet['xau'] = fuzz.trapmf(self.thoi_tiet.universe, [5, 8, 10, 10])

        self.diem_thuong['khong'] = fuzz.trimf(self.diem_thuong.universe, [0, 0, 20])
        self.diem_thuong['it'] = fuzz.trimf(self.diem_thuong.universe, [10, 30, 50])
        self.diem_thuong['trung_binh'] = fuzz.trimf(self.diem_thuong.universe, [40, 60, 80])
        self.diem_thuong['cao'] = fuzz.trimf(self.diem_thuong.universe, [70, 100, 100])

        # Tập luật (Rules 11 -> 20)
        rules = [
            ctrl.Rule(self.khach_hang['tot'] & self.dung_gio['som'], self.diem_thuong['cao']),
            ctrl.Rule(self.khach_hang['trung_binh'] & self.dung_gio['dung_gio'], self.diem_thuong['trung_binh']),
            ctrl.Rule(self.khach_hang['kem'] & self.dung_gio['tre'], self.diem_thuong['khong']),
            ctrl.Rule(self.khoang_cach['dai'] & self.giao_thong['cao'] & self.dung_gio['dung_gio'], self.diem_thuong['cao']),
            ctrl.Rule(self.khoang_cach['trung_binh'] & self.giao_thong['trung_binh'] & self.khach_hang['tot'], self.diem_thuong['trung_binh']),
            ctrl.Rule(self.khoang_cach['rat_xa'] & self.thoi_tiet['xau'] & self.khach_hang['tot'], self.diem_thuong['cao']),
            ctrl.Rule(self.khoang_cach['ngan'] & self.khach_hang['trung_binh'] & self.dung_gio['dung_gio'], self.diem_thuong['it']),
            ctrl.Rule(self.khoang_cach['dai'] & self.giao_thong['cao'] & self.dung_gio['tre'], self.diem_thuong['it']),
            ctrl.Rule(self.khoang_cach['trung_binh'] & self.thoi_tiet['trung_binh'] & self.khach_hang['tot'], self.diem_thuong['trung_binh'])
        ]

        self.ctrl_system = ctrl.ControlSystem(rules)
        self.simulator = ctrl.ControlSystemSimulation(self.ctrl_system)

    def calculate(self, kh, dg, kc, gt, tt):
        self.simulator.input['khach_hang'] = kh
        self.simulator.input['dung_gio'] = dg
        self.simulator.input['khoang_cach'] = kc
        self.simulator.input['giao_thong'] = gt
        self.simulator.input['thoi_tiet'] = tt
        self.simulator.compute()
        return self.simulator.output['diem_thuong']

display(HTML("<h2> HỆ THỐNG ĐÁNH GIÁ ĐIỂM THƯỞNG GRAB-BIKE (FUZZY LOGIC)</h2>"))

fuzzy_sys = GrabFuzzySystem()

style = {'description_width': 'initial'}
layout = widgets.Layout(width='500px')

# === ĐÃ SỬA DÒNG NÀY THÀNH NÚT BẤM ===
w_kh = widgets.ToggleButtons(
    options=['1', '2', '3', '4', '5'],
    value='4',
    description='  Đánh giá (Sao):',
    button_style='warning',
    style=style
)
# =====================================

w_dg = widgets.IntSlider(value=0, min=-15, max=15, step=1, description='Đúng giờ (-sớm / +trễ):', style=style, layout=layout)
w_kc = widgets.FloatSlider(value=10.0, min=0, max=30.0, step=0.5, description=' Khoảng cách (km):', style=style, layout=layout)
w_gt = widgets.FloatSlider(value=5.0, min=0, max=10.0, step=0.5, description=' Kẹt xe (0-10):', style=style, layout=layout)
w_tt = widgets.FloatSlider(value=2.0, min=0, max=10.0, step=0.5, description=' Thời tiết xấu (0-10):', style=style, layout=layout)

btn_calc = widgets.Button(description='TÍNH ĐIỂM THƯỞNG', button_style='success', tooltip='Click để tính toán', icon='calculator', layout=widgets.Layout(width='200px', height='40px'))
out_result = widgets.Output()

def on_click_calculate(b):
    with out_result:
        clear_output(wait=True)
        try:
            # === ĐÃ SỬA: Ép kiểu float(w_kh.value) để truyền vào dạng số thực ===
            diem = fuzzy_sys.calculate(float(w_kh.value), w_dg.value, w_kc.value, w_gt.value, w_tt.value)
            display(HTML(f"<h3 style='color: green;'> Kết quả: Điểm thưởng là <b>{diem:.2f}</b> điểm.</h3>"))
        except Exception as e:
            display(HTML(f"<h4 style='color: red;'> Cảnh báo: Các thông số hiện tại chưa kích hoạt bất kỳ bộ luật nào trong hệ thống. Hãy thử thay đổi thông số.</h4>"))

btn_calc.on_click(on_click_calculate)

# Hiển thị toàn bộ UI ra màn hình Colab
ui_box = widgets.VBox([w_kh, w_dg, w_kc, w_gt, w_tt, widgets.HTML("<br>"), btn_calc, out_result])
display(ui_box)

In [ ]:
#2.12
!pip install scikit-fuzzy -q

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML


class ShopeeFuzzySystem:
    def __init__(self):
        self.setup_system()

    def setup_system(self):

        self.xep_hang = ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'xep_hang')
        self.doanh_so = ctrl.Antecedent(np.arange(0, 1001, 1), 'doanh_so')
        self.loi_nhuan = ctrl.Antecedent(np.arange(0, 101, 1), 'loi_nhuan')
        self.su_kien = ctrl.Antecedent(np.arange(0, 11, 1), 'su_kien')
        self.canh_tranh = ctrl.Antecedent(np.arange(0, 101, 1), 'canh_tranh')

        self.chiet_khau = ctrl.Consequent(np.arange(0, 51, 1), 'chiet_khau')


        self.xep_hang['thap'] = fuzz.trimf(self.xep_hang.universe, [0, 0, 3.9])
        self.xep_hang['trung_binh'] = fuzz.trimf(self.xep_hang.universe, [3.8, 4.3, 4.6])
        self.xep_hang['cao'] = fuzz.trimf(self.xep_hang.universe, [4.5, 5, 5])

        self.doanh_so['thap'] = fuzz.trimf(self.doanh_so.universe, [0, 0, 400])
        self.doanh_so['trung_binh'] = fuzz.trimf(self.doanh_so.universe, [200, 500, 800])
        self.doanh_so['cao'] = fuzz.trimf(self.doanh_so.universe, [600, 1000, 1000])

        self.loi_nhuan['thap'] = fuzz.trimf(self.loi_nhuan.universe, [0, 0, 30])
        self.loi_nhuan['trung_binh'] = fuzz.trimf(self.loi_nhuan.universe, [20, 50, 80])
        self.loi_nhuan['cao'] = fuzz.trimf(self.loi_nhuan.universe, [60, 100, 100])

        self.su_kien['khong'] = fuzz.trimf(self.su_kien.universe, [0, 0, 4])
        self.su_kien['trung_binh'] = fuzz.trimf(self.su_kien.universe, [2, 5, 8])
        self.su_kien['cao'] = fuzz.trimf(self.su_kien.universe, [6, 10, 10])

        self.canh_tranh['thap'] = fuzz.trimf(self.canh_tranh.universe, [0, 0, 30])
        self.canh_tranh['trung_binh'] = fuzz.trimf(self.canh_tranh.universe, [20, 50, 80])
        self.canh_tranh['cao'] = fuzz.trimf(self.canh_tranh.universe, [60, 100, 100])

        self.chiet_khau['trung_binh'] = fuzz.trimf(self.chiet_khau.universe, [10, 25, 40])
        self.chiet_khau['cao'] = fuzz.trimf(self.chiet_khau.universe, [30, 40, 50])
        self.chiet_khau['rat_cao'] = fuzz.trimf(self.chiet_khau.universe, [40, 50, 50])
        self.chiet_khau['thap'] = fuzz.trimf(self.chiet_khau.universe, [0, 0, 15])


        rules = [
            ctrl.Rule(self.xep_hang['thap'] & self.su_kien['khong'], self.chiet_khau['trung_binh']),
            ctrl.Rule(self.doanh_so['thap'] & self.loi_nhuan['thap'], self.chiet_khau['rat_cao']),
            ctrl.Rule(self.xep_hang['trung_binh'] & self.doanh_so['trung_binh'] & self.loi_nhuan['thap'] & self.su_kien['cao'] & self.canh_tranh['cao'], self.chiet_khau['cao']),


            ctrl.Rule(self.doanh_so['cao'] & self.loi_nhuan['cao'], self.chiet_khau['thap']),
            ctrl.Rule(self.su_kien['cao'] & self.canh_tranh['cao'], self.chiet_khau['cao']),
            ctrl.Rule(self.doanh_so['trung_binh'] & self.loi_nhuan['trung_binh'], self.chiet_khau['trung_binh']),
            ctrl.Rule(self.su_kien['khong'] & self.canh_tranh['thap'], self.chiet_khau['thap'])
        ]

        self.ctrl_system = ctrl.ControlSystem(rules)
        self.simulator = ctrl.ControlSystemSimulation(self.ctrl_system)

    def calculate(self, xh, ds, ln, sk, ct):
        self.simulator.input['xep_hang'] = xh
        self.simulator.input['doanh_so'] = ds
        self.simulator.input['loi_nhuan'] = ln
        self.simulator.input['su_kien'] = sk
        self.simulator.input['canh_tranh'] = ct
        self.simulator.compute()
        return self.simulator.output['chiet_khau']



display(HTML("<h2> HỆ THỐNG CHIẾN LƯỢC CHIẾT KHẤU SHOPEE (BÀI 2.12)</h2>"))

shopee_sys = ShopeeFuzzySystem()

style = {'description_width': '180px'}
layout = widgets.Layout(width='600px')

w_xh = widgets.ToggleButtons(
    options=['1', '2', '3', '4', '5'],
    value='4',
    description='Xếp hạng cửa hàng:',
    button_style='warning',
    style=style
)

w_ds = widgets.IntSlider(value=500, min=0, max=1000, step=10, description='Khối lượng bán hàng:', style=style, layout=layout)
w_ln = widgets.FloatSlider(value=20.0, min=0, max=100.0, step=1, description='Biên lợi nhuận (%):', style=style, layout=layout)
w_sk = widgets.IntSlider(value=8, min=0, max=10, step=1, description='Sự kiện (Mùa sale):', style=style, layout=layout)
w_ct = widgets.FloatSlider(value=70.0, min=0, max=100.0, step=1, description='Đối thủ giảm giá (%):', style=style, layout=layout)

btn_calc = widgets.Button(description='TÍNH MỨC CHIẾT KHẤU', button_style='info', icon='percent', layout=widgets.Layout(width='200px', height='40px', margin='10px 0 0 180px'))
out_result = widgets.Output()

def on_click_calculate(b):
    with out_result:
        clear_output(wait=True)
        try:

            chiet_khau = shopee_sys.calculate(float(w_xh.value), w_ds.value, w_ln.value, w_sk.value, w_ct.value)
            display(HTML(f"""
            <div style="background-color: #fdf5e6; padding: 15px; border-radius: 8px; border-left: 5px solid #ff5722; width: 580px;">
                <h3 style='color: #ff5722; margin: 0;'> Kết luận hệ thống AI:</h3>
                <p style='font-size: 16px; margin-top: 5px;'>Mức giảm giá tối ưu đề xuất là: <b style='font-size: 22px;'>{chiet_khau:.2f}%</b></p>
            </div>
            """))
        except Exception as e:
            display(HTML(f"<h4 style='color: red;'> Cảnh báo: Các thông số hiện tại nằm ngoài vùng dữ liệu của tập luật!</h4>"))

btn_calc.on_click(on_click_calculate)

ui_box = widgets.VBox([w_xh, w_ds, w_ln, w_sk, w_ct, btn_calc, out_result])
display(ui_box)

In [5]:
#2.13
!pip install scikit-fuzzy -q
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

class ShopeePremiumFuzzy:
    def __init__(self):

        self.nhu_cau = ctrl.Antecedent(np.arange(0, 11, 1), 'nhu_cau')
        self.ap_luc = ctrl.Antecedent(np.arange(0, 101, 1), 'ap_luc')
        self.uy_tin = ctrl.Antecedent(np.arange(1, 5.1, 0.1), 'uy_tin')
        self.loi_nhuan = ctrl.Antecedent(np.arange(0, 101, 1), 'loi_nhuan')
        self.mua_vu = ctrl.Antecedent(np.arange(0, 11, 1), 'mua_vu')

        self.chiet_khau = ctrl.Consequent(np.arange(0, 71, 1), 'chiet_khau')


        self.nhu_cau.automf(3, names=['thap', 'trung_binh', 'cao'])
        self.ap_luc.automf(3, names=['thap', 'trung_binh', 'cao'])
        self.uy_tin.automf(3, names=['thap', 'trung_binh', 'cao'])
        self.loi_nhuan.automf(3, names=['thap', 'trung_binh', 'cao'])
        self.mua_vu.automf(3, names=['khong', 'trung_binh', 'cao'])

        self.chiet_khau['rat_thap'] = fuzz.trimf(self.chiet_khau.universe, [0, 2.5, 5])
        self.chiet_khau['thap'] = fuzz.trimf(self.chiet_khau.universe, [5, 7.5, 10])
        self.chiet_khau['trung_binh'] = fuzz.trimf(self.chiet_khau.universe, [10, 15, 20])
        self.chiet_khau['cao'] = fuzz.trimf(self.chiet_khau.universe, [20, 30, 40])
        self.chiet_khau['rat_cao'] = fuzz.trimf(self.chiet_khau.universe, [40, 55, 70])


        rules = [
            ctrl.Rule(self.nhu_cau['cao'] & self.ap_luc['thap'] & self.loi_nhuan['thap'], self.chiet_khau['rat_thap']),
            ctrl.Rule(self.nhu_cau['thap'] & self.ap_luc['cao'] & self.loi_nhuan['cao'], self.chiet_khau['cao']),
            ctrl.Rule(self.uy_tin['cao'] & self.loi_nhuan['trung_binh'] & self.mua_vu['cao'], self.chiet_khau['trung_binh']),
            ctrl.Rule(self.ap_luc['cao'] & self.mua_vu['cao'] & self.loi_nhuan['cao'], self.chiet_khau['rat_cao']),
            ctrl.Rule(self.uy_tin['thap'] & self.nhu_cau['trung_binh'] & self.loi_nhuan['thap'], self.chiet_khau['trung_binh']),
            ctrl.Rule(self.nhu_cau['cao'] & self.mua_vu['khong'] & self.ap_luc['thap'], self.chiet_khau['rat_thap']),
            ctrl.Rule(self.loi_nhuan['cao'] & self.ap_luc['trung_binh'] & self.mua_vu['trung_binh'], self.chiet_khau['trung_binh'])
        ]

        self.sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))

    def compute(self, nc, al, ut, ln, mv):
        self.sim.input['nhu_cau'] = nc
        self.sim.input['ap_luc'] = al
        self.sim.input['uy_tin'] = ut
        self.sim.input['loi_nhuan'] = ln
        self.sim.input['mua_vu'] = mv
        self.sim.compute()
        return self.sim.output['chiet_khau']


display(HTML("<h2>CHIẾT KHẤU SHOPEE (HÀNG ĐẶC BIỆT - BÀI 2.13)</h2>"))
sys_213 = ShopeePremiumFuzzy()

style = {'description_width': '200px'}
slider_layout = widgets.Layout(width='800px')


w_nc = widgets.FloatSlider(value=8, min=0, max=10, description='Nhu cầu (0-10):', style=style, layout=slider_layout)
w_al = widgets.FloatSlider(value=20, min=0, max=100, description=' Áp lực đối thủ (%):', style=style, layout=slider_layout)
w_ut = widgets.ToggleButtons(options=['1', '2', '3', '4', '5'], value='4', description='Uy tín (Sao):', button_style='warning', style=style)
w_ln = widgets.FloatSlider(value=80, min=0, max=100, description=' Biên lợi nhuận (%):', style=style, layout=slider_layout)
w_mv = widgets.FloatSlider(value=0, min=0, max=10, description='Sự kiện mùa vụ (0-10):', style=style, layout=slider_layout)

btn = widgets.Button(description='TÍNH CHIẾT KHẤU', button_style='info', icon='percent', layout=widgets.Layout(width='200px', height='40px', margin='10px 0 0 200px'))
out = widgets.Output()

def calc_213(b):
    with out:
        clear_output(wait=True)
        try:

            res = sys_213.compute(w_nc.value, w_al.value, float(w_ut.value), w_ln.value, w_mv.value)
            display(HTML(f"""
            <div style="background-color: #f0f8ff; padding: 15px; border-radius: 8px; border-left: 5px solid #2196f3; width: 780px;">
                <h3 style='color: #2196f3; margin: 0;'> Kết luận hệ thống AI:</h3>
                <p style='font-size: 16px; margin-top: 5px;'>Chiết khấu đề xuất cho mặt hàng này là: <b style='font-size: 22px;'>{res:.1f}%</b></p>
            </div>
            """))
        except:
            display(HTML("<h4 style='color:red;'> Không có luật phù hợp với tổ hợp thông số này.</h4>"))

btn.on_click(calc_213)
display(widgets.VBox([w_nc, w_al, w_ut, w_ln, w_mv, btn, out]))

In [6]:
#2.14
!pip install scikit-fuzzy -q

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML


class LogisticsFuzzySystem:
    def __init__(self):
        self.setup_system()

    def setup_system(self):
        self.mat_do = ctrl.Antecedent(np.arange(0, 11, 1), 'mat_do')
        self.khan_cap = ctrl.Antecedent(np.arange(0, 11, 1), 'khan_cap')
        self.tai_trong = ctrl.Antecedent(np.arange(0, 101, 1), 'tai_trong')
        self.giao_thong = ctrl.Antecedent(np.arange(0, 11, 1), 'giao_thong')
        self.loi_nhuan = ctrl.Antecedent(np.arange(0, 11, 1), 'loi_nhuan')


        self.ket_hop = ctrl.Consequent(np.arange(1, 11, 1), 'ket_hop')
        self.uu_tien = ctrl.Consequent(np.arange(0, 11, 1), 'uu_tien')


        for var in [self.mat_do, self.khan_cap, self.giao_thong, self.loi_nhuan]:
            var.automf(3, names=['thap', 'trung_binh', 'cao'])

        self.tai_trong.automf(3, names=['thap', 'trung_binh', 'cao'])


        self.ket_hop['it'] = fuzz.trimf(self.ket_hop.universe, [1, 1, 4])
        self.ket_hop['mot_so'] = fuzz.trimf(self.ket_hop.universe, [3, 5, 7])
        self.ket_hop['nhieu'] = fuzz.trimf(self.ket_hop.universe, [6, 10, 10])

        self.uu_tien['thap'] = fuzz.trimf(self.uu_tien.universe, [0, 0, 4])
        self.uu_tien['trung_binh'] = fuzz.trimf(self.uu_tien.universe, [3, 5, 7])
        self.uu_tien['cao'] = fuzz.trimf(self.uu_tien.universe, [6, 10, 10])


        rules = [
            ctrl.Rule(self.mat_do['cao'] & self.tai_trong['thap'] & self.giao_thong['thap'], self.ket_hop['nhieu']),
            ctrl.Rule(self.mat_do['trung_binh'] & self.giao_thong['cao'] & self.khan_cap['trung_binh'], self.ket_hop['mot_so']),
            ctrl.Rule(self.tai_trong['cao'] & self.mat_do['cao'] & self.loi_nhuan['trung_binh'], self.ket_hop['mot_so']),
            ctrl.Rule(self.mat_do['thap'] & self.khan_cap['cao'] & self.giao_thong['trung_binh'], self.ket_hop['mot_so']),
            ctrl.Rule(self.loi_nhuan['cao'] & self.khan_cap['cao'] & self.giao_thong['cao'], self.ket_hop['mot_so']),


            ctrl.Rule(self.khan_cap['cao'] & self.loi_nhuan['cao'], self.uu_tien['cao']),
            ctrl.Rule(self.khan_cap['trung_binh'] & self.giao_thong['trung_binh'], self.uu_tien['trung_binh']),
            ctrl.Rule(self.khan_cap['thap'] & self.mat_do['cao'] & self.loi_nhuan['thap'], self.uu_tien['thap'])
        ]

        self.sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem(rules))

    def compute(self, md, kc, tt, gt, ln):
        self.sim.input['mat_do'] = md
        self.sim.input['khan_cap'] = kc
        self.sim.input['tai_trong'] = tt
        self.sim.input['giao_thong'] = gt
        self.sim.input['loi_nhuan'] = ln
        self.sim.compute()
        return self.sim.output['ket_hop'], self.sim.output['uu_tien']

display(HTML("<h2> HỆ THỐNG ĐIỀU PHỐI LOGISTICS THÔNG MINH (BÀI 2.14)</h2>"))
logis_sys = LogisticsFuzzySystem()


style = {'description_width': '220px'}
slider_layout = widgets.Layout(width='850px')

w_md = widgets.FloatSlider(value=8, min=0, max=10, step=0.1, description='Mật độ đơn hàng (0-10):', style=style, layout=slider_layout)
w_kc = widgets.FloatSlider(value=5, min=0, max=10, step=0.1, description=' Độ khẩn cấp (0-10):', style=style, layout=slider_layout)
w_tt = widgets.FloatSlider(value=20, min=0, max=100, step=1, description=' Tải trọng xe (%):', style=style, layout=slider_layout)
w_gt = widgets.FloatSlider(value=2, min=0, max=10, step=0.1, description=' Tình trạng giao thông (0-10):', style=style, layout=slider_layout)
w_ln = widgets.FloatSlider(value=5, min=0, max=10, step=0.1, description='Lợi nhuận/đơn (0-10):', style=style, layout=slider_layout)

btn = widgets.Button(description='TỐI ƯU HÓA GIAO HÀNG', button_style='success', icon='truck', layout=widgets.Layout(width='250px', height='45px', margin='15px 0 0 220px'))
out = widgets.Output()

def process_logistics(b):
    with out:
        clear_output(wait=True)
        try:
            don_gom, diem_uu_tien = logis_sys.compute(w_md.value, w_kc.value, w_tt.value, w_gt.value, w_ln.value)


            display(HTML(f"""
            <div style="background-color: #e8f5e9; padding: 20px; border-radius: 10px; border-left: 8px solid #4caf50; width: 810px; margin-top: 20px;">
                <h3 style='color: #2e7d32; margin: 0 0 10px 0;'> KẾT QUẢ ĐIỀU PHỐI AI:</h3>
                <table style='width: 100%; font-size: 18px;'>
                    <tr>
                        <td> <b>Số đơn hàng nên gom kết hợp:</b></td>
                        <td><b style='font-size: 24px; color: #1b5e20;'>{int(don_gom)} đơn</b></td>
                    </tr>
                    <tr>
                        <td> <b>Mức độ ưu tiên giao hàng:</b></td>
                        <td><b style='font-size: 24px; color: #1b5e20;'>{diem_uu_tien:.1f} / 10</b></td>
                    </tr>
                </table>
                <p style='margin-top: 10px; font-style: italic; color: #555;'>*Dựa trên 8 luật mờ về mật độ, tải trọng và giao thông.</p>
            </div>
            """))
        except:
            display(HTML("<h4 style='color:red;'>Lỗi: Không tìm thấy luật phù hợp. Hãy thử thay đổi thông số!</h4>"))

btn.on_click(process_logistics)
ui = widgets.VBox([w_md, w_kc, w_tt, w_gt, w_ln, btn, out])
display(ui)